# Satellite Imagery Maps for Detroit Lake and Klamath Lake

This notebook creates interactive maps displaying satellite imagery from Sentinel-2, Landsat, and MODIS for Detroit Lake and Upper Klamath Lake. Each map shows recent cloud-free imagery with appropriate band combinations for water quality visualization.

In [1]:
# Import required libraries
import ee
import folium
from datetime import datetime, timedelta

# Authenticate and initialize Google Earth Engine
ee.Authenticate()
ee.Initialize(project='ee-toddsteissberg')  # Replace with your GEE project ID

In [2]:
# Define lake locations (from original notebook)
lakes = {
    'Detroit Lake': {
        'center': [44.711, -122.184],
        'point': ee.Geometry.Point([-122.184, 44.711])
    },
    'Upper Klamath Lake': {
        'center': [42.400, -121.900],
        'point': ee.Geometry.Point([-121.900, 42.400])
    }
}

# Date range for imagery (recent 6 months)
end_date = datetime.now()
start_date = end_date - timedelta(days=180)
date_range = [start_date.strftime('%Y-%m-%d'), end_date.strftime('%Y-%m-%d')]

## Sentinel-2 Maps (10m resolution)

In [3]:
def create_sentinel2_map(lake_name, lake_info):
    """Create a folium map with Sentinel-2 imagery for the specified lake."""
    
    # Get Sentinel-2 collection
    s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
        .filterDate(date_range[0], date_range[1]) \
        .filterBounds(lake_info['point']) \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)) \
        .sort('CLOUDY_PIXEL_PERCENTAGE')
    
    # Get the least cloudy image
    image = s2.first()
    
    # Visualization parameters for true color
    vis_params = {
        'min': 0,
        'max': 3000,
        'bands': ['B4', 'B3', 'B2']  # RGB
    }
    
    # Create folium map
    m = folium.Map(location=lake_info['center'], zoom_start=12)
    
    # Add the image to map
    map_id_dict = ee.Image(image).getMapId(vis_params)
    folium.TileLayer(
        tiles=map_id_dict['tile_fetcher'].url_format,
        attr='Google Earth Engine',
        overlay=True,
        name='Sentinel-2 RGB',
    ).add_to(m)
    
    # Add NDCI visualization
    ndci = image.normalizedDifference(['B5', 'B4'])  # Red-edge vs Red
    ndci_vis = {
        'min': -0.5,
        'max': 0.5,
        'palette': ['blue', 'white', 'green', 'yellow', 'red']
    }
    
    ndci_map_id = ndci.getMapId(ndci_vis)
    folium.TileLayer(
        tiles=ndci_map_id['tile_fetcher'].url_format,
        attr='Google Earth Engine',
        overlay=True,
        name='Sentinel-2 NDCI',
        show=False
    ).add_to(m)
    
    # Add layer control
    folium.LayerControl().add_to(m)
    
    # Add title
    title_html = f'''<h3 style="position: fixed; 
                     top: 10px; left: 50px; width: 400px; 
                     background-color: white; z-index: 1000; 
                     padding: 10px; border-radius: 5px;">
                     Sentinel-2: {lake_name}</h3>'''
    m.get_root().html.add_child(folium.Element(title_html))
    
    return m

# Create Sentinel-2 maps for both lakes
print("Creating Sentinel-2 map for Detroit Lake...")
s2_detroit = create_sentinel2_map('Detroit Lake', lakes['Detroit Lake'])
s2_detroit

Creating Sentinel-2 map for Detroit Lake...


In [4]:
print("Creating Sentinel-2 map for Upper Klamath Lake...")
s2_klamath = create_sentinel2_map('Upper Klamath Lake', lakes['Upper Klamath Lake'])
s2_klamath

Creating Sentinel-2 map for Upper Klamath Lake...


## Landsat Maps (30m resolution)

In [5]:
def create_landsat_map(lake_name, lake_info):
    """Create a folium map with Landsat imagery for the specified lake."""
    
    # Get Landsat 8 collection
    landsat = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2') \
        .filterDate(date_range[0], date_range[1]) \
        .filterBounds(lake_info['point']) \
        .filter(ee.Filter.lt('CLOUD_COVER', 20)) \
        .sort('CLOUD_COVER')
    
    # Get the least cloudy image
    image = landsat.first()
    
    # Apply scaling factors
    image = image.multiply(0.0000275).add(-0.2)
    
    # Visualization parameters for true color
    vis_params = {
        'min': 0,
        'max': 0.3,
        'bands': ['SR_B4', 'SR_B3', 'SR_B2']  # RGB
    }
    
    # Create folium map
    m = folium.Map(location=lake_info['center'], zoom_start=12)
    
    # Add the image to map
    map_id_dict = ee.Image(image).getMapId(vis_params)
    folium.TileLayer(
        tiles=map_id_dict['tile_fetcher'].url_format,
        attr='Google Earth Engine',
        overlay=True,
        name='Landsat RGB',
    ).add_to(m)
    
    # Add NDCI visualization
    ndci = image.normalizedDifference(['SR_B5', 'SR_B4'])  # NIR vs Red
    ndci_vis = {
        'min': -0.5,
        'max': 0.5,
        'palette': ['blue', 'white', 'green', 'yellow', 'red']
    }
    
    ndci_map_id = ndci.getMapId(ndci_vis)
    folium.TileLayer(
        tiles=ndci_map_id['tile_fetcher'].url_format,
        attr='Google Earth Engine',
        overlay=True,
        name='Landsat NDCI',
        show=False
    ).add_to(m)
    
    # Add layer control
    folium.LayerControl().add_to(m)
    
    # Add title
    title_html = f'''<h3 style="position: fixed; 
                     top: 10px; left: 50px; width: 400px; 
                     background-color: white; z-index: 1000; 
                     padding: 10px; border-radius: 5px;">
                     Landsat: {lake_name}</h3>'''
    m.get_root().html.add_child(folium.Element(title_html))
    
    return m

# Create Landsat maps for both lakes
print("Creating Landsat map for Detroit Lake...")
landsat_detroit = create_landsat_map('Detroit Lake', lakes['Detroit Lake'])
landsat_detroit

Creating Landsat map for Detroit Lake...


In [6]:
print("Creating Landsat map for Upper Klamath Lake...")
landsat_klamath = create_landsat_map('Upper Klamath Lake', lakes['Upper Klamath Lake'])
landsat_klamath

Creating Landsat map for Upper Klamath Lake...


## MODIS Maps (500m resolution)

In [7]:
def create_modis_map(lake_name, lake_info):
    """Create a folium map with MODIS Terra imagery for the specified lake."""
    
    # Get MODIS Terra collection
    modis = ee.ImageCollection('MODIS/061/MOD09GA') \
        .filterDate(date_range[0], date_range[1]) \
        .filterBounds(lake_info['point']) \
        .select(['sur_refl_b01', 'sur_refl_b04', 'sur_refl_b03', 'state_1km'])
    
    # Function to mask clouds
    def mask_clouds(img):
        qa = img.select('state_1km')
        cloud = qa.bitwiseAnd(1 << 10).eq(0)
        return img.updateMask(cloud)
    
    # Get a recent clear image
    image = modis.map(mask_clouds).median()
    
    # Apply scaling
    image = image.multiply(0.0001)
    
    # Visualization parameters for true color
    vis_params = {
        'min': 0,
        'max': 0.3,
        'bands': ['sur_refl_b01', 'sur_refl_b04', 'sur_refl_b03']  # Red, Green, Blue
    }
    
    # Create folium map (zoom out more due to coarser resolution)
    m = folium.Map(location=lake_info['center'], zoom_start=10)
    
    # Add the image to map
    map_id_dict = ee.Image(image).getMapId(vis_params)
    folium.TileLayer(
        tiles=map_id_dict['tile_fetcher'].url_format,
        attr='Google Earth Engine',
        overlay=True,
        name='MODIS RGB',
    ).add_to(m)
    
    # Add chlorophyll visualization using green/red ratio
    green_red_ratio = image.select('sur_refl_b04').divide(image.select('sur_refl_b01'))
    chl_vis = {
        'min': 0.5,
        'max': 3,
        'palette': ['blue', 'cyan', 'green', 'yellow', 'orange', 'red']
    }
    
    chl_map_id = green_red_ratio.getMapId(chl_vis)
    folium.TileLayer(
        tiles=chl_map_id['tile_fetcher'].url_format,
        attr='Google Earth Engine',
        overlay=True,
        name='MODIS Green/Red Ratio',
        show=False
    ).add_to(m)
    
    # Add layer control
    folium.LayerControl().add_to(m)
    
    # Add title
    title_html = f'''<h3 style="position: fixed; 
                     top: 10px; left: 50px; width: 400px; 
                     background-color: white; z-index: 1000; 
                     padding: 10px; border-radius: 5px;">
                     MODIS Terra: {lake_name}</h3>'''
    m.get_root().html.add_child(folium.Element(title_html))
    
    # Add marker for lake center
    folium.Marker(
        lake_info['center'],
        popup=f"{lake_name} Center",
        icon=folium.Icon(color='blue', icon='info-sign')
    ).add_to(m)
    
    return m

# Create MODIS maps for both lakes
print("Creating MODIS map for Detroit Lake...")
modis_detroit = create_modis_map('Detroit Lake', lakes['Detroit Lake'])
modis_detroit

Creating MODIS map for Detroit Lake...


In [8]:
print("Creating MODIS map for Upper Klamath Lake...")
modis_klamath = create_modis_map('Upper Klamath Lake', lakes['Upper Klamath Lake'])
modis_klamath

Creating MODIS map for Upper Klamath Lake...


## Summary

This notebook has created 6 interactive maps showing satellite imagery for Detroit Lake and Upper Klamath Lake:

1. **Sentinel-2 Detroit Lake** - 10m resolution, RGB and NDCI layers
2. **Sentinel-2 Upper Klamath Lake** - 10m resolution, RGB and NDCI layers
3. **Landsat Detroit Lake** - 30m resolution, RGB and NDCI layers
4. **Landsat Upper Klamath Lake** - 30m resolution, RGB and NDCI layers
5. **MODIS Detroit Lake** - 500m resolution, RGB and Green/Red ratio layers
6. **MODIS Upper Klamath Lake** - 500m resolution, RGB and Green/Red ratio layers

Each map includes:
- True color RGB visualization
- Water quality index visualization (NDCI for Sentinel/Landsat, Green/Red ratio for MODIS)
- Layer control to toggle between visualizations
- Appropriate zoom levels for each sensor's resolution

The maps use the most recent cloud-free imagery available within the last 6 months.